# Addition Transformer (~10M params)

Train a character-level decoder-only transformer on **simple addition** using only **JAX + Flax + Optax**

- **Vocabulary:** digits `0-9`, space, `+`, `=` (plus PAD)
- **Data:** random `a + b = c` with `a, b` in `[0, 999]`
- **Model:** ~10M parameters, fixed sequence length 20
- **Runtime:** ~15-25 minutes on Colab **T4 GPU** (15k steps, curriculum to 3-digit addition)
- **Roofline:** single-GPU Colab; T4 critical intensity is ~217 FLOPs/byte in fp16 (65 TFLOP/s / 300 GB/s) but only ~27 in fp32 - which regime you are in decides whether kernel fusion can pay off (see section 5)

## 0. Setup (Colab)

1. **Runtime -> Change runtime type -> T4 GPU** (recommended, A100 also works)
2. Run cells in order. The setup cell installs deps and puts the repo root on `sys.path`

**Get the code on to Colab** (pick one)
- **Upload** Files panel -> upload the whole `jax-pretraining` folder, then open this notebook from inside it
- **Git clone** set `REPO_URL` in the setup cell to your fork (must include `addition_transformer/`, `moe/`, `kernels/`, `chinchilla/`, and `benchmarks/`)

In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# --- Option B: git clone (set your fork URL, then re-run) ---
REPO_URL = "https://github.com/rktreddy/jax-pretraining.git"

# Colab ships a working CUDA jax - do NOT reinstall it (that kills the kernel).
# Only add what's missing.
missing = [m for m in ("flax", "optax", "orbax.checkpoint") if importlib.util.find_spec(m) is None]
if missing:
    !pip -q install flax optax orbax-checkpoint

ROOT = Path.cwd()
if not (ROOT / "addition_transformer").exists():
    alt = ROOT / "jax-pretraining"
    if (alt / "addition_transformer").exists():
        ROOT = alt
    elif REPO_URL:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, "jax-pretraining"])
        ROOT = Path("jax-pretraining")
    else:
        raise FileNotFoundError(
            "Cannot find addition_transformer/. Upload the repo folder to Colab "
            "or set REPO_URL above to your git clone URL"
        )

os.chdir(ROOT)
sys.path.insert(0, str(Path.cwd()))  # flat-layout repo: import packages straight from the repo root
print("Project root:", Path.cwd())

Project root: /content/jax-pretraining


In [2]:
import jax
devs = jax.devices()
dev = devs[0]
print("Devices:", devs)
print(f"Platform: {dev.platform} | {dev.device_kind}")

# Rough bf16/fp16 matmul alpha (Ch.1 / Ch.12) - single device only
_ALPHA_HINTS = {
    "t4": "~200",
    "a100": "~156",
    "h100": "~295", 
}
kind = dev.device_kind.lower()
alpha = next((v for k, v in _ALPHA_HINTS.items() if k in kind), None)
if dev.platform == "gpu" and alpha:
    print(f"Approx matmul alpha {alpha} | addition FFN effective B~128 -> memory-bound")
elif dev.platform == "cpu":
    print(f"Warning: CPU runtime - training works but is much slower. Switch to T4 GPU.")
if len(devs) == 1:
    print("Note: single GPU - ch12 NVLink collectives (450 GB/s) are not exercised here.")

Devices: [CudaDevice(id=0)]
Platform: gpu | Tesla T4
Approx matmul alpha ~200 | addition FFN effective B~128 -> memory-bound
Note: single GPU - ch12 NVLink collectives (450 GB/s) are not exercised here.


## 1. Quick sanity checks

In [3]:
from addition_transformer.data import format_example, generate_batch
from addition_transformer.vocab import decode, encode, VOCAB_SIZE, MAX_SEQ_LEN, CHARS
import jax

print('Chars', CHARS)
print('Vocab size', VOCAB_SIZE, '| Max seq len', MAX_SEQ_LEN)
ex = format_example(123, 456)
print('Example:', ex)
print('Encoded:', encode(ex))
print('Decoded:', decode(encode(ex)))

key = jax.random.key(0)
ids, labels, answer_mask = generate_batch(key, 4)
for i in range(4):
    print(decode(ids[i]))

Chars ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ' ', '+', '=']
Vocab size 14 | Max seq len 20
Example: 123 + 456 = 579
Encoded: [2, 3, 4, 11, 12, 11, 5, 6, 7, 11, 13, 11, 6, 8, 10, 0, 0, 0, 0, 0]
Decoded: 123 + 456 = 579
791 + 22 = 813
561 + 22 = 583
447 + 496 = 943
417 + 135 = 552


## 2. Train — ~15k steps, curriculum to 3-digit addition

In [4]:
from addition_transformer.train import TrainConfig, train

config = TrainConfig(
    batch_size=512,
    max_steps=15000,
    log_every=100,
    eval_every=500,
    learnin_rate=3e-4,
    save_checkpoint_at_end=False  # skip disk writing on ephemeral Colab WM
)
state = train(config)

Model parameters: (9869440,)(9.87M)
 curriculum | max_operans=9
step   100 | loss 0.0001 | answer_acc 1.000 | max_op 9
step   200 | loss 0.0000 | answer_acc 1.000 | max_op 9
step   300 | loss 0.0000 | answer_acc 1.000 | max_op 9
step   400 | loss 0.0000 | answer_acc 1.000 | max_op 9
step   500 | loss 0.0000 | answer_acc 1.000 | max_op 9
 eval | loss 8.7298 | answer_tok_acc 0.125 | addition_acc 0.000
  OK  '7 + 8 = 15'  (expected '7 + 8 = 15')
  FAIL  '48 + 58 = '  (expected '48 + 58 = 106')
  FAIL  '123 + 456 = '  (expected '123 + 456 = 579')
  FAIL  '999 + 999 = 1111'  (expected '999 + 999 = 1998')
step   600 | loss 0.0000 | answer_acc 1.000 | max_op 9
step   700 | loss 0.0000 | answer_acc 1.000 | max_op 9
step   800 | loss 0.0000 | answer_acc 1.000 | max_op 9
step   900 | loss 0.0000 | answer_acc 1.000 | max_op 9
step  1000 | loss 0.0000 | answer_acc 1.000 | max_op 9
 eval | loss 9.0499 | answer_tok_acc 0.126 | addition_acc 0.000
  OK  '7 + 8 = 15'  (expected '7 + 8 = 15')
  FAIL  '4

## 3. Try some prompts

In [5]:
from addition_transformer.data import format_example
from addition_transformer.train import generate_fixed

prompts = [(0, 0), (7, 8), (42, 58), (123, 456), (999, 1), (999, 999)]
for a, b in prompts:
    p = f"{a} + {b} = "
    expected = format_example(a, b)
    out = generate_fixed(state, p)
    ok = "OK" if out == expected else "FAIL"
    print(f"{ok}  {p!r} -> {out!r}. (expected  {expected!r})")

OK  '0 + 0 = ' -> '0 + 0 = 0'. (expected  '0 + 0 = 0')
OK  '7 + 8 = ' -> '7 + 8 = 15'. (expected  '7 + 8 = 15')
FAIL  '42 + 58 = ' -> '42 + 58 = 108'. (expected  '42 + 58 = 100')
OK  '123 + 456 = ' -> '123 + 456 = 579'. (expected  '123 + 456 = 579')
FAIL  '999 + 1 = ' -> '999 + 1 = 999'. (expected  '999 + 1 = 1000')
OK  '999 + 999 = ' -> '999 + 999 = 1998'. (expected  '999 + 999 = 1998')


## 4. Parameter count (roofline sanity check)

A ~10M model on addition is massively over-parameterized - that's intentional. Later exercises derive Chinchilla scaling laws and compare dense vs MoE

In [6]:
from addition_transformer.model import count_params

n = count_params(state.params)
print(f"Parameters: {n:,} {n/1e6:.2f}M")
print("FFN matmul intensity ~batch_size=512 -> still memory-bound on T4 (alpha=200)")

Parameters: 9,869,440 9.87M
FFN matmul intensity ~batch_size=512 -> still memory-bound on T4 (alpha=200)


## 5. (Optional) MoE benchmark on this GPU

Runs `benchmarks/benchmark_moe.py` with platform/roofline context.

Note: the default shapes (`D=512, F=4096`, fp32) are **compute-bound** on a T4 (intensity >> 27 FLOPs/byte), so the fused kernel cannot beat `ragged_dot` here no matter how good it is - the H round trip it saves is hidden behind the compute wall. To see a real fusion win, you need the unfused version memory-bound: cast inputs to fp16 and use small `D` with `F/D >= 8`, e.g. `--tokens 65536 --d 64 --f 1024` (predicted ~3.5x from the roofline: unfused moves ~287 MB vs ~19 MB fused, against a 0.26 ms compute floor).

In [7]:
!PYTHONPATH=. python benchmarks/benchmark_moe.py --tokens 4096 --d 512 --f 4096 --experts 8 --iters 10

Platform: gpu | Tesla T4 | devices: 1
  Book approx: ~300 GB/s HBM, ~65 TFLOPs fp16 TC, matmul alpha pprox ~200
.  Single GPU: ch12 NVLink/IB collectives (450 GB/s) not exercised.


Benchmark: tokens=4096, D=512, F=4096, G=8, F/D=8.0, f_tile=512

method            ms/call    speedup
------------------------------------
E0710 19:24:07.433237   22457 slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %dot = f32[4096,4096]{1,0} dot(%constant.2, %constant.4), lhs_contracting_dims={1}, rhs_contracting_dims={0}, metadata={op_name="jit(<lambda>)/dot_general" source_file="/content/jax-pretraining/moe/ragged_moe.py" source_line=48 source_end_line=48 source_column=23 source_end_column=34}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that take

## 6. protect against the runtime evaporating

In [8]:
from addition_transformer.checkpoint import save_checkpoint
save_checkpoint(state, config, "checkpoints/run1")
!zip -qr run1.zip checkpoints/run1

Saved checkpoint to /content/jax-pretraining/checkpoints/run1
